# NGSO SLS - Slice A: Coverage Explorer

Interactive coverage/availability analysis for NGSO constellations (analytic Kepler+J2, H3 grid, single-owner sharding). Presets cover the **Reliance-Jio sizing scenarios**; **Custom (Walker)** gives full T/P/F + altitude + inclination for up to two shells.

**k (min sats in view)** = minimum number of satellites simultaneously above the min elevation for a cell to count as covered; **availability** = fraction of time that holds.

Run the cells top to bottom. Adjust **Controls**, then click **Run simulation** (or re-run the **Results** cell). Result plots appear in tabs, over country borders.

## Setup

In [ ]:
# === Setup: make ngso_sls importable (Colab + local Jupyter Lab) ===
# LOCAL JUPYTER (recommended): in a terminal, `pip install -e .` in the repo once, then this
#   cell is a no-op (it detects ngso_sls and skips clone/install entirely).
# COLAB: set REPO_URL to your remote. Private repo -> add a GitHub token in Colab 'Secrets'
#   named GITHUB_TOKEN (enable Notebook access).
# NOTE: after you push new code, do Runtime -> Restart runtime, then Run all — a running kernel
#   keeps already-imported modules, so code changes only take effect on a fresh kernel.
REPO_URL = "https://github.com/luca-aalyria/spacetime-sls.git"

import importlib, importlib.util, subprocess, sys, os, re


def _run(cmd):
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if p.returncode != 0:
        print("$", cmd)
        print(p.stdout[-2000:])
        print(p.stderr[-3000:])
        raise RuntimeError(f"command failed (exit {p.returncode}) - see output above")


if importlib.util.find_spec("ngso_sls") is None:  # no-op on local Jupyter if already installed
    url = REPO_URL
    try:  # optional private-repo auth via Colab secret GITHUB_TOKEN
        from google.colab import userdata
        _tok = userdata.get("GITHUB_TOKEN")
        if _tok and url.startswith("https://github.com/"):
            url = url.replace("https://", f"https://{_tok}@")
    except Exception:
        pass
    repo_dir = re.sub(r"\.git$", "", os.path.basename(REPO_URL.rstrip("/"))) or "repo"
    if not os.path.isdir(repo_dir):
        _run(f"git clone {url} {repo_dir}")
    else:
        _run(f"git -C {repo_dir} pull --ff-only")  # update a stale clone on a fresh kernel
    _run(f"{sys.executable} -m pip install {os.path.abspath(repo_dir)}")
    sys.path.insert(0, os.path.abspath(repo_dir))
    importlib.invalidate_caches()

import ngso_sls
print("ngso_sls", ngso_sls.__version__)

## Controls
Set parameters here. `Global` at high H3 resolution is heavy - start coarse.

In [ ]:
from ngso_sls.explorer import CoverageExplorer
from IPython.display import display

explorer = CoverageExplorer()
display(explorer.controls)

## Results
Click **Run simulation** (or run this cell). Tabs: coverage availability map, mean-satellites-in-view map, satellites-in-view vs latitude, availability histogram. A `coverage_availability.csv` is also written.

In [ ]:
display(explorer.results)
explorer.run()   # initial run; also re-runs when you click "Run simulation" above